# Training dan Evaluasi Sistem Rekomendasi Parfum

Notebook ini adalah tahap eksperimen utama sebelum aplikasi Flask dibuat.
Alurnya:

1. Load `Datasets/final_df.csv`.
2. Data understanding dan preprocessing.
3. Training model TF-IDF untuk fitur aroma parfum.
4. Evaluasi model similarity menggunakan metrik retrieval Top-K.
5. Implementasi rule dan Fuzzy SAW.
6. Uji skenario rekomendasi.
7. Simpan dataset bersih, model, matrix, dan metrik evaluasi untuk dipakai Flask.

Catatan metodologi: project ini adalah hybrid recommender, bukan klasifikasi supervised.
Karena dataset tidak memiliki label "cocok/tidak cocok", evaluasi model memakai proxy label
`fragrance_family` dan metrik retrieval seperti Precision@K, Recall@K, HitRate@K, MRR@K,
dan nDCG@K.

## 1. Import Library dan Konfigurasi Path

In [1]:
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_PATH = PROJECT_ROOT / "Datasets" / "final_df.csv"
PROCESSED_PATH = PROJECT_ROOT / "Datasets" / "perfume_clean.csv"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

VECTORIZER_PATH = MODEL_DIR / "tfidf_vectorizer.joblib"
MATRIX_PATH = MODEL_DIR / "tfidf_matrix.joblib"
METADATA_PATH = MODEL_DIR / "metadata.json"
EVALUATION_PATH = REPORT_DIR / "model_evaluation.json"

DATASET_PATH

WindowsPath('F:/perkuliahan/Semester 6/SPK/project/parfum/Perfumes_Recommender-main/Datasets/final_df.csv')

## 2. Load Dataset Mentah

In [2]:
df_raw = pd.read_csv(DATASET_PATH)
df_raw.shape

(4239, 18)

In [3]:
df_raw.head()

,Name,Price,Description,Rate,Rating_count,image,Brand,Gender,Product_Type,Character_x,Fragrance_Family,Size,Year,Ingredients,Concentration,Top_note,Middle_note,Base_note
0,Dolce & Gabanna L'imperatrice 3 Pour Femme,199,Perfume for the energetic woman who is a hero in her movie in life every day! It keeps you vibrant and sparkling wit...,5,6,https://assets.goldenscent.com/catalog/product/cache/1/image/9df78eab33525d08d6e5fb8d27136e95/3/4/3423473020615-dolc...,Dolce&Gabbana,Women,Perfume,Romantic,Floral,100 ml,2009,"watermerlon, kiwi, pink cyclamen, musk, pink pepper, jasmine, sandalwood, lemon tree",Eau de Toilette,"pink pepper, kiwi, rhubarb","jasmine, cyclamen, watermelon","musk, sandalwood, lemon trees."
1,Roberto Cavalli Paradiso,169,"Woody floral fragrance, a subtle aroma that makes you feel fresh with a stunning blend of citrus, jasmine, and cypress.",4.95,17,https://assets.goldenscent.com/catalog/product/cache/1/image/9df78eab33525d08d6e5fb8d27136e95/3/6/3607347733423c-rob...,Roberto Cavalli,Women,Perfume,Romantic,Woody,50 ml,2015,"citrus, mandarin, bergamot, jasmine, pine, cypress, laurel.",Eau de Parfum,"citruses , mandarin , bergamot",jasmine,"cypress, parasol pine, pink laurel"
2,Yves Saint Laurent Libre,389,"This perfume is a reflection of Freedom, specially designed for bold women, for those who live by their own rules, t...",5,3,https://assets.goldenscent.com/catalog/product/cache/1/image/9df78eab33525d08d6e5fb8d27136e95/3/6/3614272648418c-yve...,Yves Saint Laurent,Women,Perfume,Romantic,Floral,90 ml,2019,"mandarin orange, lavendar, black currant, petitgrain, jasmine, orange blossom, vanilla, cedar, musk, ambergris",Eau de Parfum,"mandarin orange, lavendar, black currant, petitgrain","jasmine, orange blossom","vanilla, cedar, musk, ambergris"
3,Mancera Red Tobacco,499,"Mancera Red Tobacco is the new oriental, woody fragrance for men and women. Its the best oriental scent, full of lif...",4.38,8,https://assets.goldenscent.com/catalog/product/cache/1/image/9df78eab33525d08d6e5fb8d27136e95/3/6/3607347733423c-man...,Mancera,Women,Perfume,Romantic,Oriental,120 ml,2017,"saffron, cinnamon, incense, nutmeg, white peach, green apple & nepalese oud, leaves of patchouli, delicate jasmine, ...",Eau de Parfum,"saffron, cinnamon, incense, nutmeg, white peach, green apple, nepalese","patchouli, jasmine","tobacco, amber, woody notes, vetiver, vanilla, white musk"
4,Giorgio Armani Emporio Armani Stronger With You Intensely,399,"for every romantic gentle man, This strong fascinating fragrance is for you. It was released to capture all people h...",5,3,https://assets.goldenscent.com/catalog/product/cache/1/image/9df78eab33525d08d6e5fb8d27136e95/3/6/3614272225718-gior...,Giorgio Armani,Men,Perfume,Romantic,Aromatic,100 ml,2019,"spices, violet, lavender, sweet toffee, caramel, cinnamon, suede, vanilla, amber",Eau de Parfum,"pink pepper, juniper, violet leaf","lavender, sage, toffee, cinnamon","tonka bean, suede, amber, vanilla"


In [4]:
pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "unique": df_raw.nunique(dropna=True),
})

,dtype,missing,unique
Name,object,0,3525
Price,int64,0,876
Description,object,0,3566
Rate,object,0,22
Rating_count,object,0,13
image,object,0,3090
Brand,object,0,385
Gender,object,0,4
Product_Type,object,0,15
Character_x,object,0,12


## 3. Data Understanding Ringkas

Kolom yang penting untuk model:

- `Name`, `Brand`, `Gender`, `Product_Type`
- `Price`, `Rate`, `Rating_count`
- `Fragrance_Family`, `Character_x`
- `Top_note`, `Middle_note`, `Base_note`, `Ingredients`, `Description`

In [5]:
display(df_raw["Product_Type"].fillna("NA").value_counts().head(15))
display(df_raw["Gender"].fillna("NA").value_counts())
display(df_raw["Fragrance_Family"].fillna("NA").value_counts().head(20))
display(df_raw["Rate"].fillna("NA").value_counts().head(15))

Product_Type
Perfume               3530
Perfume Set            357
Perfume Oil             72
Hair Mist               62
Room Spray              62
Body Mist               43
Bakhoor                 36
Oud                     28
Hair & Body Mist        15
Diffuser                15
Candle                   8
Bundles                  5
Body Lotion              4
Perfume                  1
Home Fragrance Set       1
Name: count, dtype: int64

Gender
Women     2167
Unisex    1095
Men        862
Home       115
Name: count, dtype: int64

Fragrance_Family
Floral               1256
Oriental              683
Woody                 596
Aromatic              404
Fruity                332
Citrus                293
Floral Oriental       196
Woody Oriental        133
Leather                89
Oud                    84
Soft Floral            55
Aquatic                51
Soft Oriental          28
Arabian                22
Chypre                  7
Sweet                   3
Floral                  2
Floral,Fruity           2
Floral,Woody            2
Aromatic,Oriental       1
Name: count, dtype: int64

Rate
none    3832
5        299
4         34
4.5       22
3          9
4.75       5
1          5
3.5        5
4.25       4
4.83       4
4.6        4
4.67       3
3.67       3
4.88       2
4.38       1
Name: count, dtype: int64

## 4. Fungsi Preprocessing

In [6]:
def clean_text(value):
    if value is None:
        return ""
    text = str(value).strip()
    if text.lower() in {"nan", "none", "null"}:
        return ""
    return text


def normalize_text(value):
    text = clean_text(value).lower()
    text = re.sub(r"[^a-z0-9,\s&-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_float(value, default=0.0):
    try:
        if value is None or str(value).strip() == "":
            return default
        return float(str(value).replace(",", "."))
    except (TypeError, ValueError):
        return default


def normalize_gender(value):
    text = normalize_text(value)
    if text in {"men", "man", "male", "pria", "laki laki"}:
        return "Men"
    if text in {"women", "woman", "female", "wanita", "perempuan"}:
        return "Women"
    if text in {"unisex", "all", "semua"}:
        return "Unisex"
    return clean_text(value).title()


def tokenize_terms(*values):
    terms = set()
    for value in values:
        text = normalize_text(value)
        if not text:
            continue
        parts = re.split(r"[,/&\-\s]+", text)
        terms.update(part for part in parts if len(part) > 2)
    return terms

In [7]:
def preprocess_perfume_data(raw_df):
    df = raw_df.rename(columns={
        "Name": "name",
        "Price": "price",
        "Description": "description",
        "Rate": "rating",
        "Rating_count": "rating_count",
        "Brand": "brand",
        "Gender": "gender",
        "Product_Type": "product_type",
        "Character_x": "character",
        "Fragrance_Family": "fragrance_family",
        "Size": "size",
        "Year": "year",
        "Ingredients": "ingredients",
        "Concentration": "concentration",
        "Top_note": "top_note",
        "Middle_note": "middle_note",
        "Base_note": "base_note",
    }).copy()

    text_columns = [
        "name", "description", "image", "brand", "gender", "product_type",
        "character", "fragrance_family", "size", "ingredients",
        "concentration", "top_note", "middle_note", "base_note",
    ]
    for column in text_columns:
        if column not in df.columns:
            df[column] = ""
        df[column] = df[column].map(clean_text)

    df["price"] = pd.to_numeric(df.get("price", 0), errors="coerce").fillna(0.0)
    df["rating"] = pd.to_numeric(df.get("rating", 0), errors="coerce").fillna(0.0)
    df["rating_count"] = pd.to_numeric(df.get("rating_count", 0), errors="coerce").fillna(0).astype(int)
    df["year"] = pd.to_numeric(df.get("year", 0), errors="coerce").fillna(0).astype(int)
    df["gender"] = df["gender"].map(normalize_gender)

    product_mask = df["product_type"].str.contains("perfume", case=False, na=False)
    gender_mask = df["gender"].ne("Home")
    df = df[product_mask & gender_mask].copy()

    df = df.drop_duplicates(subset=["name", "brand", "size"], keep="first")
    df = df.reset_index(drop=True)
    df["perfume_id"] = df.index
    df["fragrance_family"] = df["fragrance_family"].str.replace(r"\s+", " ", regex=True).str.strip()

    df["aroma_profile"] = (
        df["fragrance_family"] + " " +
        df["character"] + " " +
        df["top_note"] + " " +
        df["middle_note"] + " " +
        df["base_note"] + " " +
        df["ingredients"] + " " +
        df["description"]
    ).map(normalize_text)

    df["display_price"] = df["price"].map(lambda value: f"{value:,.2f}".rstrip("0").rstrip("."))
    df["display_rating"] = df["rating"].map(
        lambda value: f"{value:.2f}".rstrip("0").rstrip(".") if value > 0 else "Belum ada"
    )
    return df


df = preprocess_perfume_data(df_raw)
df.shape

(3296, 22)

In [8]:
df[[
    "perfume_id", "name", "brand", "gender", "product_type", "price",
    "rating", "fragrance_family", "aroma_profile"
]].head()

,perfume_id,name,brand,gender,product_type,price,rating,fragrance_family,aroma_profile
0,0,Dolce & Gabanna L'imperatrice 3 Pour Femme,Dolce&Gabbana,Women,Perfume,199,5.00,Floral,"floral romantic pink pepper, kiwi, rhubarb jasmine, cyclamen, watermelon musk, sandalwood, lemon trees watermerlon, ..."
1,1,Roberto Cavalli Paradiso,Roberto Cavalli,Women,Perfume,169,4.95,Woody,"woody romantic citruses , mandarin , bergamot jasmine cypress, parasol pine, pink laurel citrus, mandarin, bergamot,..."
2,2,Yves Saint Laurent Libre,Yves Saint Laurent,Women,Perfume,389,5.00,Floral,"floral romantic mandarin orange, lavendar, black currant, petitgrain jasmine, orange blossom vanilla, cedar, musk, a..."
3,3,Mancera Red Tobacco,Mancera,Women,Perfume,499,4.38,Oriental,"oriental romantic saffron, cinnamon, incense, nutmeg, white peach, green apple, nepalese patchouli, jasmine tobacco,..."
4,4,Giorgio Armani Emporio Armani Stronger With You Intensely,Giorgio Armani,Men,Perfume,399,5.00,Aromatic,"aromatic romantic pink pepper, juniper, violet leaf lavender, sage, toffee, cinnamon tonka bean, suede, amber, vanil..."


## 5. EDA Setelah Cleaning

In [9]:
display(df["gender"].value_counts())
display(df["fragrance_family"].value_counts().head(20))
display(df[["price", "rating", "rating_count"]].describe())

gender
Women     1785
Unisex     814
Men        697
Name: count, dtype: int64

fragrance_family
Floral               1020
Oriental              503
Woody                 501
Aromatic              300
Fruity                241
Citrus                227
Floral Oriental       162
Woody Oriental        100
Leather                71
Oud                    43
Soft Floral            42
Aquatic                38
Soft Oriental          23
Arabian                17
Chypre                  3
Floral,Fruity           2
Sweet                   1
Floral,Woody            1
Aromatic,Oriental       1
Name: count, dtype: int64

,price,rating,rating_count
count,3296.000000,3296.000000,3296.000000
mean,519.910498,0.454378,0.187500
std,496.100450,1.407349,0.839489
min,25.000000,0.000000,0.000000
25%,208.000000,0.000000,0.000000
50%,380.000000,0.000000,0.000000
75%,653.000000,0.000000,0.000000
max,8193.000000,5.000000,18.000000


## 6. Training Model TF-IDF

Pada model TF-IDF, proses training adalah `fit` terhadap corpus teks.
Corpus yang digunakan adalah `aroma_profile`, yaitu gabungan fragrance family,
character, top notes, middle notes, base notes, ingredients, dan description.

Untuk evaluasi offline, data dibagi menjadi train/test. Setelah evaluasi selesai,
vectorizer final akan di-fit ulang pada seluruh dataset bersih dan disimpan untuk Flask.

In [10]:
evaluable_df = df[df["fragrance_family"].ne("")].copy()
family_counts = evaluable_df["fragrance_family"].value_counts()
evaluable_df = evaluable_df[evaluable_df["fragrance_family"].isin(family_counts[family_counts >= 3].index)].copy()

train_df, test_df = train_test_split(
    evaluable_df,
    test_size=0.2,
    random_state=42,
    stratify=evaluable_df["fragrance_family"],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

tfidf_eval = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    strip_accents="unicode",
)
train_matrix = tfidf_eval.fit_transform(train_df["aroma_profile"])

train_df.shape, test_df.shape, train_matrix.shape

((2632, 22), (659, 22), (2632, 68024))

## 7. Evaluasi Model Similarity

Karena tidak ada label kecocokan user-parfum, evaluasi memakai proxy:

- Query test dibuat dari `character`, `top_note`, `middle_note`, `base_note`, dan `ingredients`.
- Item dianggap relevan jika memiliki `fragrance_family` yang sama.
- Hasil dibandingkan dengan random baseline.

Metrik:

- Precision@K: proporsi hasil Top-K yang relevan.
- Recall@K: proporsi item relevan yang berhasil ditemukan.
- HitRate@K: minimal satu item relevan muncul di Top-K.
- MRR@K: posisi relevan pertama.
- nDCG@K: kualitas urutan ranking.

In [11]:
def dcg_at_k(relevance):
    relevance = np.asarray(relevance, dtype=float)
    if relevance.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, relevance.size + 2))
    return float(np.sum(relevance / discounts))


def metrics_for_indices(retrieved_indices, relevant_mask, k):
    top_indices = np.asarray(retrieved_indices[:k])
    rel = relevant_mask[top_indices].astype(int)
    hits = int(rel.sum())
    total_relevant = int(relevant_mask.sum())
    precision = hits / k
    recall = hits / total_relevant if total_relevant else 0.0
    hit_rate = 1.0 if hits > 0 else 0.0

    relevant_positions = np.where(rel == 1)[0]
    mrr = 1.0 / (relevant_positions[0] + 1) if len(relevant_positions) else 0.0

    ideal_relevance = np.ones(min(total_relevant, k))
    ndcg = dcg_at_k(rel) / dcg_at_k(ideal_relevance) if total_relevant else 0.0
    return precision, recall, hit_rate, mrr, ndcg


def evaluate_similarity_model(train_df, test_df, vectorizer, train_matrix, k_values=(5, 10), max_queries=500, seed=42):
    rng = np.random.default_rng(seed)
    sample_df = test_df.sample(n=min(max_queries, len(test_df)), random_state=seed).reset_index(drop=True)
    rows = []

    for _, row in sample_df.iterrows():
        query = normalize_text(
            " ".join([
                row["character"],
                row["top_note"],
                row["middle_note"],
                row["base_note"],
                row["ingredients"],
            ])
        )
        if not query:
            continue

        query_vector = vectorizer.transform([query])
        scores = cosine_similarity(query_vector, train_matrix).flatten()
        ranked_indices = np.argsort(scores)[::-1]
        relevant_mask = train_df["fragrance_family"].eq(row["fragrance_family"]).to_numpy()

        random_indices = rng.permutation(len(train_df))

        for k in k_values:
            precision, recall, hit_rate, mrr, ndcg = metrics_for_indices(ranked_indices, relevant_mask, k)
            rows.append({
                "model": "tfidf_cosine",
                "k": k,
                "precision": precision,
                "recall": recall,
                "hit_rate": hit_rate,
                "mrr": mrr,
                "ndcg": ndcg,
            })

            precision, recall, hit_rate, mrr, ndcg = metrics_for_indices(random_indices, relevant_mask, k)
            rows.append({
                "model": "random_baseline",
                "k": k,
                "precision": precision,
                "recall": recall,
                "hit_rate": hit_rate,
                "mrr": mrr,
                "ndcg": ndcg,
            })

    return pd.DataFrame(rows)


evaluation_rows = evaluate_similarity_model(train_df, test_df, tfidf_eval, train_matrix)
evaluation_summary = (
    evaluation_rows
    .groupby(["model", "k"], as_index=False)
    .mean(numeric_only=True)
    .sort_values(["k", "model"])
)
evaluation_summary

,model,k,precision,recall,hit_rate,mrr,ndcg
0,random_baseline,5,0.1584,0.002032,0.516,0.291800,0.159775
2,tfidf_cosine,5,0.2824,0.003534,0.618,0.409000,0.282683
1,random_baseline,10,0.1692,0.004449,0.728,0.320679,0.166981
3,tfidf_cosine,10,0.2638,0.006572,0.790,0.432621,0.269157


In [12]:
evaluation_pivot = evaluation_summary.pivot(index="k", columns="model", values=["precision", "recall", "hit_rate", "mrr", "ndcg"])
evaluation_pivot

precision                       recall               \
model random_baseline tfidf_cosine random_baseline tfidf_cosine   
k                                                                 
5              0.1584       0.2824        0.002032     0.003534   
10             0.1692       0.2638        0.004449     0.006572   

             hit_rate                          mrr               \
model random_baseline tfidf_cosine random_baseline tfidf_cosine   
k                                                                 
5               0.516        0.618        0.291800     0.409000   
10              0.728        0.790        0.320679     0.432621   

                 ndcg               
model random_baseline tfidf_cosine  
k                                   
5            0.159775     0.282683  
10           0.166981     0.269157

## 8. Training Final dan Simpan Artifact Model

In [13]:
final_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    strip_accents="unicode",
)
final_tfidf_matrix = final_vectorizer.fit_transform(df["aroma_profile"])

df.to_csv(PROCESSED_PATH, index=False)
joblib.dump(final_vectorizer, VECTORIZER_PATH)
joblib.dump(final_tfidf_matrix, MATRIX_PATH)

metadata = {
    "dataset_source": str(DATASET_PATH.relative_to(PROJECT_ROOT)),
    "processed_dataset": str(PROCESSED_PATH.relative_to(PROJECT_ROOT)),
    "vectorizer": str(VECTORIZER_PATH.relative_to(PROJECT_ROOT)),
    "tfidf_matrix": str(MATRIX_PATH.relative_to(PROJECT_ROOT)),
    "n_items": int(len(df)),
    "n_features": int(final_tfidf_matrix.shape[1]),
    "training_method": "TfidfVectorizer.fit_transform(aroma_profile)",
    "evaluation_method": "Top-K retrieval with fragrance_family as proxy relevance label",
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

metadata

{'dataset_source': 'Datasets\\final_df.csv',
 'processed_dataset': 'Datasets\\perfume_clean.csv',
 'vectorizer': 'models\\tfidf_vectorizer.joblib',
 'tfidf_matrix': 'models\\tfidf_matrix.joblib',
 'n_items': 3296,
 'n_features': 79937,
 'training_method': 'TfidfVectorizer.fit_transform(aroma_profile)',
 'evaluation_method': 'Top-K retrieval with fragrance_family as proxy relevance label'}

## 9. Rule SPK dan Fuzzy SAW

In [14]:
WEIGHTS = {
    "gender_score": 0.20,
    "age_score": 0.15,
    "aroma_similarity": 0.25,
    "rating_score": 0.15,
    "price_score": 0.10,
    "family_score": 0.15,
}

AGE_RULES = [
    (0, 20, {"fresh", "citrus", "fruity", "sweet"}),
    (21, 30, {"fresh", "aquatic", "floral", "woody", "sweet"}),
    (31, 45, {"woody", "spicy", "amber", "musk", "aromatic", "oud"}),
    (46, 200, {"oriental", "leather", "amber", "powdery", "musk", "oud"}),
]

RELATED_FAMILIES = [
    {"fresh", "citrus", "aquatic", "fruity", "green"},
    {"floral", "soft", "sweet", "fruity", "romantic"},
    {"woody", "aromatic", "spicy", "oud", "leather"},
    {"oriental", "amber", "musk", "arabian", "soft", "oud"},
    {"sweet", "vanilla", "gourmand", "fruity", "caramel"},
]

USAGE_KEYWORDS = {
    "daily": "fresh citrus aquatic clean light",
    "campus": "fresh citrus fruity aquatic light",
    "work": "woody floral musk aromatic clean elegant",
    "formal": "woody amber musk oriental sophisticated",
    "date": "sweet floral musk vanilla romantic warm",
}


def has_related_overlap(left, right):
    for group in RELATED_FAMILIES:
        if (left & group) and (right & group):
            return True
    return False


def get_age_terms(age):
    for minimum, maximum, terms in AGE_RULES:
        if minimum <= age <= maximum:
            return terms
    return AGE_RULES[-1][2]


def gender_score(user_gender, perfume_gender):
    user = normalize_gender(user_gender)
    perfume = normalize_gender(perfume_gender)
    if user == perfume:
        return 1.0
    if perfume == "Unisex":
        return 0.8
    if user == "Unisex" and perfume in {"Men", "Women"}:
        return 0.7
    return 0.3


def age_score(age, fragrance_family, aroma_profile):
    age_terms = get_age_terms(age)
    perfume_terms = tokenize_terms(fragrance_family, aroma_profile)
    if age_terms & perfume_terms:
        return 1.0
    if has_related_overlap(age_terms, perfume_terms):
        return 0.7
    return 0.4


def rating_score(rating):
    value = parse_float(rating, 0.0)
    if value <= 0:
        return 0.0
    return min(value / 5.0, 1.0)


def price_score(price, budget):
    value = parse_float(price, 0.0)
    if budget <= 0:
        return 0.7
    if value <= budget:
        return 1.0
    if value <= budget * 1.2:
        return 0.7
    return 0.3


def family_score(preferred_family, aroma_keywords, row_family, aroma_profile):
    preferred = normalize_text(preferred_family)
    row = normalize_text(row_family)
    perfume_terms = tokenize_terms(row_family, aroma_profile)
    preferred_terms = tokenize_terms(preferred_family, aroma_keywords)
    if preferred and (preferred == row or preferred in row or row in preferred):
        return 1.0
    if preferred_terms and preferred_terms & perfume_terms:
        return 0.85
    if preferred_terms and has_related_overlap(preferred_terms, perfume_terms):
        return 0.7
    if preferred:
        return 0.3
    return 0.6


def fuzzy_saw_score(scores):
    return round(sum(scores[key] * weight for key, weight in WEIGHTS.items()), 4)

## 10. Fungsi Rekomendasi Hybrid

In [15]:
def build_query(preference):
    age_terms = " ".join(sorted(get_age_terms(preference["age"])))
    usage_terms = USAGE_KEYWORDS.get(normalize_text(preference.get("usage", "daily")), "")
    parts = [
        preference.get("fragrance_family", ""),
        preference.get("aroma_keywords", ""),
        usage_terms,
        age_terms,
    ]
    query = " ".join(part for part in parts if clean_text(part))
    return normalize_text(query or "fresh floral woody citrus musk")


def make_reason(row, preference):
    reasons = []
    if row["gender_score"] >= 0.8:
        reasons.append(f"gender {row['gender']} sesuai")
    if row["age_score"] >= 0.9:
        reasons.append("profil aroma cocok dengan kelompok umur")
    if row["family_score"] >= 0.8:
        reasons.append(f"family {row['fragrance_family']} relevan")
    if row["aroma_similarity"] >= 0.25:
        reasons.append("notes aromanya mirip dengan input")
    if row["rating_score"] >= 0.8:
        reasons.append("rating tinggi")
    if row["price_score"] >= 0.9 and preference["budget"] > 0:
        reasons.append("harga masuk budget")
    if not reasons:
        reasons.append("skor gabungan ML dan SPK paling kompetitif")
    return ", ".join(reasons).capitalize() + "."


def recommend_perfume(
    gender="Unisex",
    age=25,
    fragrance_family="",
    aroma_keywords="",
    budget=0,
    minimal_rating=0,
    usage="daily",
    top_n=10,
):
    preference = {
        "gender": normalize_gender(gender),
        "age": int(age),
        "fragrance_family": clean_text(fragrance_family),
        "aroma_keywords": clean_text(aroma_keywords),
        "budget": float(budget),
        "minimal_rating": float(minimal_rating),
        "usage": clean_text(usage) or "daily",
        "top_n": int(top_n),
    }

    candidates = df.copy()
    if preference["minimal_rating"] > 0:
        filtered = candidates[candidates["rating"].ge(preference["minimal_rating"])].copy()
        if not filtered.empty:
            candidates = filtered

    query = build_query(preference)
    query_vector = final_vectorizer.transform([query])
    candidate_matrix = final_tfidf_matrix[candidates.index]
    candidates = candidates.assign(aroma_similarity=cosine_similarity(query_vector, candidate_matrix).flatten())

    score_rows = []
    for _, row in candidates.iterrows():
        scores = {
            "gender_score": gender_score(preference["gender"], row["gender"]),
            "age_score": age_score(preference["age"], row["fragrance_family"], row["aroma_profile"]),
            "aroma_similarity": float(row["aroma_similarity"]),
            "rating_score": rating_score(row["rating"]),
            "price_score": price_score(row["price"], preference["budget"]),
            "family_score": family_score(
                preference["fragrance_family"],
                preference["aroma_keywords"],
                row["fragrance_family"],
                row["aroma_profile"],
            ),
        }
        score_rows.append(scores | {"final_score": fuzzy_saw_score(scores)})

    score_df = pd.DataFrame(score_rows, index=candidates.index)
    ranked = pd.concat([candidates, score_df.drop(columns=["aroma_similarity"])], axis=1)
    ranked = ranked.sort_values(
        by=["final_score", "aroma_similarity", "rating", "rating_count"],
        ascending=[False, False, False, False],
    ).head(preference["top_n"]).copy()
    ranked["rank"] = range(1, len(ranked) + 1)
    ranked["reason"] = ranked.apply(lambda row: make_reason(row, preference), axis=1)

    columns = [
        "rank", "name", "brand", "gender", "price", "rating", "rating_count",
        "fragrance_family", "character", "top_note", "middle_note", "base_note",
        "aroma_similarity", "gender_score", "age_score", "rating_score",
        "price_score", "family_score", "final_score", "reason",
    ]
    return ranked[columns]

## 11. Evaluasi Fuzzy SAW dan Rule Consistency

In [16]:
# Verifikasi formula: final_score harus sama dengan jumlah skor komponen dikali bobot.
sample_scores = {
    "gender_score": 1.0,
    "age_score": 1.0,
    "aroma_similarity": 0.8,
    "rating_score": 0.9,
    "price_score": 1.0,
    "family_score": 0.85,
}
manual_score = sum(sample_scores[key] * WEIGHTS[key] for key in WEIGHTS)
function_score = fuzzy_saw_score(sample_scores)

consistency_checks = pd.DataFrame([
    {
        "test": "Formula Fuzzy SAW",
        "expected": round(manual_score, 4),
        "actual": function_score,
        "passed": round(manual_score, 4) == function_score,
    },
    {
        "test": "Gender match lebih tinggi dari gender mismatch",
        "expected": True,
        "actual": gender_score("Men", "Men") > gender_score("Men", "Women"),
        "passed": gender_score("Men", "Men") > gender_score("Men", "Women"),
    },
    {
        "test": "Harga dalam budget lebih tinggi dari jauh di atas budget",
        "expected": True,
        "actual": price_score(400, 500) > price_score(800, 500),
        "passed": price_score(400, 500) > price_score(800, 500),
    },
    {
        "test": "Rating 5 lebih tinggi dari rating kosong",
        "expected": True,
        "actual": rating_score(5) > rating_score(0),
        "passed": rating_score(5) > rating_score(0),
    },
])
consistency_checks

,test,expected,actual,passed
0,Formula Fuzzy SAW,0.9125,0.9125,True
1,Gender match lebih tinggi dari gender mismatch,True,True,True
2,Harga dalam budget lebih tinggi dari jauh di atas budget,True,True,True
3,Rating 5 lebih tinggi dari rating kosong,True,True,True


## 12. Uji Skenario Rekomendasi

In [17]:
scenario_1 = recommend_perfume(
    gender="Men",
    age=22,
    fragrance_family="Citrus",
    aroma_keywords="fresh citrus aquatic",
    budget=500,
    minimal_rating=4.0,
    usage="campus",
    top_n=5,
)
scenario_1

,rank,name,brand,gender,price,rating,rating_count,fragrance_family,character,top_note,middle_note,base_note,aroma_similarity,gender_score,age_score,rating_score,price_score,family_score,final_score,reason
311,1,Who Am I Just Right EDP,The Different Company,Men,149,5.0,3,Citrus,Dynamic,bergamot,marine acqua,oakmoss,0.040833,1.0,1.0,1.0,1.0,1.00,0.7602,"Gender men sesuai, profil aroma cocok dengan kelompok umur, family citrus relevan, rating tinggi, harga masuk budget."
1152,2,Mercedes Benz Cologne,Mercedes Benz,Men,357,5.0,1,Citrus,Romantic,"grapefruit, mandarin, pink pepper, brazilian orange","ginger, verena, fresh notes","woods, vetiver, musk.",0.040053,1.0,1.0,1.0,1.0,1.00,0.7600,"Gender men sesuai, profil aroma cocok dengan kelompok umur, family citrus relevan, rating tinggi, harga masuk budget."
205,3,Giorgio Armani Acqua Di Gio Homme,Giorgio Armani,Men,365,5.0,1,Aromatic,Romantic,"jasmine, rosemary, hespiradic notes","persimmon fruits, marine notes","cedar, patchouli, white musk, rock rose",0.121259,1.0,1.0,1.0,1.0,0.85,0.7578,"Gender men sesuai, profil aroma cocok dengan kelompok umur, family aromatic relevan, rating tinggi, harga masuk budget."
233,4,Cartier Pasha Noire,Cartier,Men,461,5.0,1,Fruity,Romantic,green citrus,woody amber,cedar accords,0.046187,1.0,1.0,1.0,1.0,0.85,0.7390,"Gender men sesuai, profil aroma cocok dengan kelompok umur, family fruity relevan, rating tinggi, harga masuk budget."
1136,5,Carolina Herrera Chic,Carolina Herrera,Men,233,5.0,1,Aquatic,Romantic,"freshness of water and juices, sicilian bergamot, ice-cold lemon, italian mandarin, fresh melon",cinnamon and black pepper,"sandalwood, cedar, ambergris, musk and tonka bean.",0.041531,1.0,1.0,1.0,1.0,0.85,0.7379,"Gender men sesuai, profil aroma cocok dengan kelompok umur, family aquatic relevan, rating tinggi, harga masuk budget."


In [18]:
scenario_2 = recommend_perfume(
    gender="Women",
    age=25,
    fragrance_family="Floral",
    aroma_keywords="floral sweet musk",
    budget=700,
    minimal_rating=4.0,
    usage="work",
    top_n=5,
)
scenario_2

,rank,name,brand,gender,price,rating,rating_count,fragrance_family,character,top_note,middle_note,base_note,aroma_similarity,gender_score,age_score,rating_score,price_score,family_score,final_score,reason
134,1,Lorenzo Villoresi Firenze Teint De Neige EDT,Lorenzo Villoresi Firenze,Women,659,5.0,1,Floral,Romantic,"jasmin, rose, ylang ylang, sweet, powdery, floral notes","tonka bean, jasmin, rose, sweet, powdery, floral notes","heliotrope, musk, rose, jasmin, sweet, powdery, floral notes",0.075642,1.0,1.0,1.0,1.0,1.0,0.7689,"Gender women sesuai, profil aroma cocok dengan kelompok umur, family floral relevan, rating tinggi, harga masuk budget."
251,2,Mancera Coco Vanille,Mancera,Women,399,5.0,1,Floral,Feminine,"coconut, white peach,","jasmine,ylang-ylang, tiare flower","white musk, woody notes, madagascar vanilla",0.071630,1.0,1.0,1.0,1.0,1.0,0.7679,"Gender women sesuai, profil aroma cocok dengan kelompok umur, family floral relevan, rating tinggi, harga masuk budget."
63,3,Candy Addict Powdery Pow EDP,Candy Addict,Women,139,5.0,1,Floral Oriental,Romantic,osmanthus,musk,amber,0.056129,1.0,1.0,1.0,1.0,1.0,0.7640,"Gender women sesuai, profil aroma cocok dengan kelompok umur, family floral oriental relevan, rating tinggi, harga m..."
380,4,Jean Paul Gaultier Scandal,Jean Paul Gaultier,Women,440,5.0,1,Floral,Glamorous,"blood orange, grapefruit","honey, gardenia.",patchouli,0.050425,1.0,1.0,1.0,1.0,1.0,0.7626,"Gender women sesuai, profil aroma cocok dengan kelompok umur, family floral relevan, rating tinggi, harga masuk budget."
61,5,Narciso Rodriguez Pure Musc For Her,Narciso Rodriguez,Women,385,5.0,2,Floral,Romantic,"jasmine, orange blossom",musk,"cashmeran, amber, patchouli",0.046894,1.0,1.0,1.0,1.0,1.0,0.7617,"Gender women sesuai, profil aroma cocok dengan kelompok umur, family floral relevan, rating tinggi, harga masuk budget."


In [19]:
scenario_3 = recommend_perfume(
    gender="Unisex",
    age=35,
    fragrance_family="Woody",
    aroma_keywords="woody amber spicy",
    budget=1000,
    minimal_rating=4.0,
    usage="formal",
    top_n=5,
)
scenario_3

,rank,name,brand,gender,price,rating,rating_count,fragrance_family,character,top_note,middle_note,base_note,aroma_similarity,gender_score,age_score,rating_score,price_score,family_score,final_score,reason
188,1,Gucci Oud,Gucci,Unisex,386,5.0,1,Woody Oriental,Charismatic,"pear, raspberry, saffron","bulgarian rose, orange blossom","patchouli, amber, musk, oud",0.092323,1.0,1.0,1.0,1.0,1.00,0.7731,"Gender unisex sesuai, profil aroma cocok dengan kelompok umur, family woody oriental relevan, rating tinggi, harga m..."
115,2,Guerlain Oud Essentiel,Guerlain,Unisex,862,5.0,2,Woody,Charismatic,"saffron, geranium","oud, bulgarian rose, atlas cedar","leather, frankincense, gaiac wood",0.032487,1.0,1.0,1.0,1.0,1.00,0.7581,"Gender unisex sesuai, profil aroma cocok dengan kelompok umur, family woody relevan, rating tinggi, harga masuk budget."
1138,3,Mood By Minis Amber Mood EDP,Minis,Unisex,40,5.0,1,Woody,Romantic,apple blossom,"amber, ylang-ylang",cedarwood,0.028596,1.0,1.0,1.0,1.0,1.00,0.7571,"Gender unisex sesuai, profil aroma cocok dengan kelompok umur, family woody relevan, rating tinggi, harga masuk budget."
94,4,Van Cleef & Arpels Precious Oud Collection Extraordinaire,Van Cleef & Arpels,Unisex,782,5.0,1,Woody,Charismatic,"bergamot, and pink pepper","jasmine, tuberose and incense","vetiver, sandalwood, patchouli, agarwood and ambergris",0.023961,1.0,1.0,1.0,1.0,1.00,0.7560,"Gender unisex sesuai, profil aroma cocok dengan kelompok umur, family woody relevan, rating tinggi, harga masuk budget."
34,5,Remal Oud Wood Bakhoor,Remal,Unisex,125,5.0,2,Oriental,Charismatic,agarwood,musk,amber,0.064858,1.0,1.0,1.0,1.0,0.85,0.7437,"Gender unisex sesuai, profil aroma cocok dengan kelompok umur, family oriental relevan, rating tinggi, harga masuk b..."


## 13. Simpan Laporan Evaluasi

In [20]:
evaluation_report = {
    "dataset": {
        "raw_rows": int(len(df_raw)),
        "clean_rows": int(len(df)),
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "n_tfidf_features": int(final_tfidf_matrix.shape[1]),
    },
    "retrieval_metrics": evaluation_summary.to_dict(orient="records"),
    "consistency_checks": consistency_checks.to_dict(orient="records"),
    "scenario_top_1": {
        "scenario_1": scenario_1.iloc[0][["name", "brand", "final_score"]].to_dict(),
        "scenario_2": scenario_2.iloc[0][["name", "brand", "final_score"]].to_dict(),
        "scenario_3": scenario_3.iloc[0][["name", "brand", "final_score"]].to_dict(),
    },
}

EVALUATION_PATH.write_text(json.dumps(evaluation_report, indent=2), encoding="utf-8")
EVALUATION_PATH

WindowsPath('F:/perkuliahan/Semester 6/SPK/project/parfum/Perfumes_Recommender-main/reports/model_evaluation.json')

## 14. Kesimpulan Eksperimen

Notebook ini menghasilkan artifact yang dipakai oleh Flask:

- `Datasets/perfume_clean.csv`
- `models/tfidf_vectorizer.joblib`
- `models/tfidf_matrix.joblib`
- `models/metadata.json`
- `reports/model_evaluation.json`

Setelah notebook ini dijalankan, aplikasi Flask dapat memuat model hasil training,
menghitung cosine similarity dari input user, lalu melakukan ranking akhir dengan Fuzzy SAW.